# Bank Ticket Priority Router English demo
Upload the complete repository ZIP or GitHub Download ZIP, then run cells in order. Saved single examples and final metrics require no key. CSV batches are live only. Keep the runtime running during recording; the shared page is temporary.


In [ ]:
from google.colab import files
import os, pathlib, zipfile, sys, subprocess
uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Upload one complete repository ZIP.')
destination = pathlib.Path('/content/bank_ticket_submission')
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    for name in archive.namelist():
        parts = pathlib.PurePosixPath(name)
        if parts.is_absolute() or '..' in parts.parts or '\\' in name:
            raise ValueError('Archive has an unsafe path.')
    archive.extractall(destination)
candidates = [p.parent for p in destination.rglob('app.py')
              if (p.parent/'ticket_router.py').is_file() and (p.parent/'data/test.csv').is_file()]
if len(candidates) != 1:
    raise ValueError('Expected one project with app.py, ticket_router.py and data/test.csv. Use a fresh runtime if needed.')
os.chdir(candidates[0])
sys.path.insert(0, str(pathlib.Path.cwd()))
print('Repository loaded:', pathlib.Path.cwd().name)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('UI dependencies installed.')


## Hidden configuration
Enter an OpenRouter key for live CSV classification, or leave it blank for saved outputs. Choose a web password. Username: hannah.


In [ ]:
from getpass import getpass
api_key = getpass('OpenRouter key (blank for saved examples only): ').strip()
if api_key:
    if not api_key.isascii() or any(ord(c) <= 32 or ord(c) == 127 for c in api_key):
        raise ValueError('Paste only the key, without spaces or non-ASCII characters.')
    os.environ['OPENROUTER_API_KEY'] = api_key
else:
    os.environ.pop('OPENROUTER_API_KEY', None)
del api_key
web_password = getpass('Set a web login password: ')
if not web_password:
    raise ValueError('A password is required for a shared demo.')
print('Configured. Web username: hannah. Credentials are not written to files.')


In [ ]:
if 'demo' in globals():
    demo.close()
for module_name in ('app', 'demo_core', 'ticket_router'):
    sys.modules.pop(module_name, None)
from app import launch_demo
demo = launch_demo(share=True, password=web_password, prevent_thread_lock=True)
